In [7]:
import io
import os
import re
from typing import Optional

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv


def make_s3_client():
    load_dotenv()

    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )


def list_klines_day_keys(
    date: str,
    bucket: str = "binance-data-downloader",
    raw_prefix: str = "raw",
    interval: str = "1m",
    symbol: Optional[str] = None,
    s3_client=None,
) -> list[str]:
    s3 = s3_client or make_s3_client()
    source_prefix = f"{raw_prefix.strip('/')}/klines/"
    day_suffix = f"/interval={interval}/date={date}/data.parquet"
    symbol_part = f"symbol={symbol}/" if symbol else None

    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith(day_suffix):
                continue
            if symbol_part and symbol_part not in key:
                continue
            keys.append(key)

    return sorted(keys)


def find_first_klines_day(
    bucket: str = "binance-data-downloader",
    raw_prefix: str = "raw",
    interval: str = "1m",
    s3_client=None,
) -> str:
    s3 = s3_client or make_s3_client()
    source_prefix = f"{raw_prefix.strip('/')}/klines/"
    pattern = re.compile(rf"/interval={re.escape(interval)}/date=(\d{{4}}-\d{{2}}-\d{{2}})/data\.parquet$")

    dates = set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            match = pattern.search(obj["Key"])
            if match:
                dates.add(match.group(1))

    if not dates:
        raise FileNotFoundError(f"No klines parquet files found under s3://{bucket}/{source_prefix}")

    return sorted(dates)[0]


def build_log_return_1m_for_day(
    date: str,
    bucket: str = "binance-data-downloader",
    raw_prefix: str = "raw",
    interval: str = "1m",
    symbol: Optional[str] = None,
    s3_client=None,
) -> pd.DataFrame:
    """Load one raw klines day from S3 and build minute log returns.

    Returns columns: date, symbol, timestamp, log_return_1m.
    If symbol is omitted, the function builds the feature for every symbol
    found for this date and concatenates the result into one daily DataFrame.
    """
    s3 = s3_client or make_s3_client()
    keys = list_klines_day_keys(
        date=date,
        bucket=bucket,
        raw_prefix=raw_prefix,
        interval=interval,
        symbol=symbol,
        s3_client=s3,
    )

    if not keys:
        raise FileNotFoundError(f"No klines data found for date={date}, interval={interval}, symbol={symbol}")

    frames = []
    symbol_pattern = re.compile(r"/symbol=([^/]+)/")

    for key in keys:
        obj = s3.get_object(Bucket=bucket, Key=key)
        df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

        symbol_match = symbol_pattern.search(f"/{key}")
        key_symbol = symbol_match.group(1) if symbol_match else symbol

        if df.empty:
            continue

        df = df.sort_values("timestamp").copy()
        df["log_return_1m"] = np.log(df["close"].astype("float64")).diff()

        feature_df = df[["timestamp", "log_return_1m"]].copy()
        feature_df.insert(0, "symbol", key_symbol)
        feature_df.insert(0, "date", date)
        frames.append(feature_df)

    if not frames:
        return pd.DataFrame(columns=["date", "symbol", "timestamp", "log_return_1m"])

    return pd.concat(frames, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)


In [8]:
s3 = make_s3_client()
first_day = find_first_klines_day(s3_client=s3)
log_return_1m = build_log_return_1m_for_day(first_day, s3_client=s3)

print(first_day)
print(log_return_1m.shape)
display(log_return_1m.head())
display(log_return_1m.groupby("symbol")["log_return_1m"].agg(["count", "mean", "std", "min", "max"]))


2020-02-01
(2878, 4)


,date,symbol,timestamp,log_return_1m
0,2020-02-01,ADAUSDT,2020-02-01 00:01:00+00:00,NaN
1,2020-02-01,ADAUSDT,2020-02-01 00:02:00+00:00,-0.000744
2,2020-02-01,ADAUSDT,2020-02-01 00:03:00+00:00,0.000186
3,2020-02-01,ADAUSDT,2020-02-01 00:04:00+00:00,-0.001861
4,2020-02-01,ADAUSDT,2020-02-01 00:05:00+00:00,-0.000745


,count,mean,std,min,max
symbol,,,,,
ADAUSDT,1438,0.000030,0.001408,-0.010935,0.012075
BTCUSDT,1438,0.000001,0.000498,-0.002746,0.003856


In [9]:
FEATURE_SYMBOLS = ["ADAUSDT"]
BUCKET = "binance-data-downloader"
RAW_PREFIX = "raw"
FEATURES_PREFIX = "features"
FEATURE_NAME = "return_1m_forward"
INTERVAL = "1m"
SKIP_EXISTING = True


def list_symbol_klines_days(
    symbol: str,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
    s3_client=None,
) -> list[str]:
    s3 = s3_client or make_s3_client()
    source_prefix = f"{raw_prefix.strip('/')}/klines/symbol={symbol}/interval={interval}/"
    pattern = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")

    dates = set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            match = pattern.search(f"/{obj['Key']}")
            if match:
                dates.add(match.group(1))

    return sorted(dates)


def read_symbol_klines_day(
    symbol: str,
    date: str,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
    s3_client=None,
) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    key = f"{raw_prefix.strip('/')}/klines/symbol={symbol}/interval={interval}/date={date}/data.parquet"
    obj = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

    if df.empty:
        return pd.DataFrame(columns=["timestamp", "close"])

    return (
        df[["timestamp", "close"]]
        .assign(
            timestamp=lambda x: pd.to_datetime(x["timestamp"], utc=True),
            close=lambda x: x["close"].astype("float64"),
        )
        .dropna(subset=["timestamp", "close"])
        .drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


def build_forward_return_1m_for_symbol_day(
    symbol: str,
    date: str,
    next_date: Optional[str] = None,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
    s3_client=None,
) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    df = read_symbol_klines_day(symbol, date, bucket, raw_prefix, interval, s3)

    if df.empty:
        return pd.DataFrame(columns=["date", "symbol", "timestamp", "return_1m"])

    if next_date is not None:
        next_df = read_symbol_klines_day(symbol, next_date, bucket, raw_prefix, interval, s3)
        if not next_df.empty:
            df = pd.concat([df, next_df.head(1)], ignore_index=True)

    df = df.sort_values("timestamp").reset_index(drop=True)
    df["next_timestamp"] = df["timestamp"].shift(-1)
    df["next_close"] = df["close"].shift(-1)
    df["return_1m"] = df["next_close"] / df["close"] - 1.0

    one_minute = pd.Timedelta(minutes=1)
    valid_next_minute = df["next_timestamp"].sub(df["timestamp"]).eq(one_minute)
    feature_df = df.loc[valid_next_minute, ["timestamp", "return_1m"]].copy()
    feature_df.insert(0, "symbol", symbol)
    feature_df.insert(0, "date", date)

    return feature_df.reset_index(drop=True)


def feature_return_key(
    symbol: str,
    date: str,
    features_prefix: str = FEATURES_PREFIX,
    feature_name: str = FEATURE_NAME,
    interval: str = INTERVAL,
) -> str:
    return (
        f"{features_prefix.strip('/')}/{feature_name}/"
        f"symbol={symbol}/interval={interval}/date={date}/data.parquet"
    )


def s3_key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except Exception as exc:
        error_code = getattr(exc, "response", {}).get("Error", {}).get("Code")
        if error_code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise


def write_returns_for_symbol_to_s3(
    symbol: str,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    features_prefix: str = FEATURES_PREFIX,
    feature_name: str = FEATURE_NAME,
    interval: str = INTERVAL,
    skip_existing: bool = SKIP_EXISTING,
    s3_client=None,
) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    dates = list_symbol_klines_days(symbol, bucket, raw_prefix, interval, s3)

    if not dates:
        raise FileNotFoundError(f"No klines days found for symbol={symbol}, interval={interval}")

    rows = []
    for i, date in enumerate(dates):
        key = feature_return_key(symbol, date, features_prefix, feature_name, interval)
        if skip_existing and s3_key_exists(s3, bucket, key):
            print(f"Skip exists: s3://{bucket}/{key}")
            rows.append({"symbol": symbol, "date": date, "rows": None, "key": key, "status": "skipped"})
            continue

        next_date = dates[i + 1] if i + 1 < len(dates) else None
        feature_df = build_forward_return_1m_for_symbol_day(
            symbol=symbol,
            date=date,
            next_date=next_date,
            bucket=bucket,
            raw_prefix=raw_prefix,
            interval=interval,
            s3_client=s3,
        )

        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())

        print(f"Uploaded: s3://{bucket}/{key} rows={len(feature_df)}")
        rows.append({"symbol": symbol, "date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

    return pd.DataFrame(rows)


# s3 = make_s3_client()
# write_results = pd.concat(
#     [write_returns_for_symbol_to_s3(symbol, s3_client=s3) for symbol in FEATURE_SYMBOLS],
#     ignore_index=True,
# )

# display(write_results)


In [10]:
def build_forward_return_n_minutes_for_symbol_day(
    symbol: str,
    date: str,
    n_minutes: int,
    next_dates: Optional[list[str]] = None,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
    s3_client=None,
) -> pd.DataFrame:
    """Build forward simple returns between minute t and t + n_minutes.

    Rows without an exact timestamp t + n_minutes are dropped. Pass the next
    available dates to compute the last n minutes of the current day when the
    target timestamp lives in the following day.
    """
    if n_minutes <= 0:
        raise ValueError("n_minutes must be positive")

    s3 = s3_client or make_s3_client()
    current_day = read_symbol_klines_day(symbol, date, bucket, raw_prefix, interval, s3)

    if current_day.empty:
        return pd.DataFrame(columns=["date", "symbol", "timestamp", f"return_{n_minutes}m"])

    frames = [current_day]
    for next_date in next_dates or []:
        next_day = read_symbol_klines_day(symbol, next_date, bucket, raw_prefix, interval, s3)
        if not next_day.empty:
            frames.append(next_day)

        last_timestamp = frames[-1]["timestamp"].max()
        if last_timestamp >= current_day["timestamp"].max() + pd.Timedelta(minutes=n_minutes):
            break

    prices = (
        pd.concat(frames, ignore_index=True)
        .drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    target = prices[["timestamp", "close"]].rename(
        columns={"timestamp": "target_timestamp", "close": "target_close"}
    )

    feature_df = current_day[["timestamp", "close"]].copy()
    feature_df["target_timestamp"] = feature_df["timestamp"] + pd.Timedelta(minutes=n_minutes)
    feature_df = feature_df.merge(target, on="target_timestamp", how="inner")
    feature_df[f"return_{n_minutes}m"] = feature_df["target_close"] / feature_df["close"] - 1.0

    feature_df = feature_df[["timestamp", f"return_{n_minutes}m"]].copy()
    feature_df.insert(0, "symbol", symbol)
    feature_df.insert(0, "date", date)

    return feature_df.reset_index(drop=True)


# # Example: 15-minute forward returns for one day.
# n_minutes = 15
# symbol = "ADAUSDT"
# dates = list_symbol_klines_days(symbol, s3_client=s3)
# date_idx = dates.index(first_day) if first_day in dates else 0
# return_15m = build_forward_return_n_minutes_for_symbol_day(
#     symbol=symbol,
#     date=dates[date_idx],
#     n_minutes=n_minutes,
#     next_dates=dates[date_idx + 1 : date_idx + 3],
#     s3_client=s3,
# )

# print(dates[date_idx], return_15m.shape)
# display(return_15m.head())
# display(return_15m.tail())


In [11]:
RETURN_HORIZONS_MINUTES = [15, 20, 25]


def write_forward_returns_n_minutes_for_symbol_to_s3(
    symbol: str,
    n_minutes: int,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    features_prefix: str = FEATURES_PREFIX,
    interval: str = INTERVAL,
    skip_existing: bool = SKIP_EXISTING,
    s3_client=None,
) -> pd.DataFrame:
    """Build and upload one forward-return feature directory for one symbol.

    The output path is:
    features/return_{n_minutes}m_forward/symbol=.../interval=1m/date=.../data.parquet
    """
    s3 = s3_client or make_s3_client()
    dates = list_symbol_klines_days(symbol, bucket, raw_prefix, interval, s3)

    if not dates:
        raise FileNotFoundError(f"No klines days found for symbol={symbol}, interval={interval}")

    feature_name = f"return_{n_minutes}m_forward"
    rows = []

    for i, date in enumerate(dates):
        key = feature_return_key(
            symbol=symbol,
            date=date,
            features_prefix=features_prefix,
            feature_name=feature_name,
            interval=interval,
        )
        if skip_existing and s3_key_exists(s3, bucket, key):
            print(f"Skip exists: s3://{bucket}/{key}")
            rows.append(
                {
                    "symbol": symbol,
                    "horizon_minutes": n_minutes,
                    "date": date,
                    "rows": None,
                    "key": key,
                    "status": "skipped",
                }
            )
            continue

        feature_df = build_forward_return_n_minutes_for_symbol_day(
            symbol=symbol,
            date=date,
            n_minutes=n_minutes,
            next_dates=dates[i + 1 :],
            bucket=bucket,
            raw_prefix=raw_prefix,
            interval=interval,
            s3_client=s3,
        )

        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())

        print(f"Uploaded: s3://{bucket}/{key} rows={len(feature_df)}")
        rows.append(
            {
                "symbol": symbol,
                "horizon_minutes": n_minutes,
                "date": date,
                "rows": len(feature_df),
                "key": key,
                "status": "uploaded",
            }
        )

    return pd.DataFrame(rows)


def write_forward_returns_for_horizons_to_s3(
    symbols: list[str] = FEATURE_SYMBOLS,
    horizons_minutes: list[int] = RETURN_HORIZONS_MINUTES,
    s3_client=None,
) -> pd.DataFrame:
    """Upload forward-return features for all requested symbols and horizons."""
    s3 = s3_client or make_s3_client()
    results = []

    for n_minutes in horizons_minutes:
        for symbol in symbols:
            results.append(
                write_forward_returns_n_minutes_for_symbol_to_s3(
                    symbol=symbol,
                    n_minutes=n_minutes,
                    s3_client=s3,
                )
            )

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


# Build and upload 15m, 20m and 25m forward returns into three directories under features/.
s3 = make_s3_client()
forward_returns_write_results = write_forward_returns_for_horizons_to_s3(s3_client=s3)

display(forward_returns_write_results)


Uploaded: s3://binance-data-downloader/features/return_15m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet rows=1439
Uploaded: s3://binance-data-downloader/features/return_15m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-02/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/return_15m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-03/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/return_15m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-04/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/return_15m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-05/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/return_15m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-06/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/return_15m_forward/symbol=ADAUSDT/interval=1m/date=2020-02-07/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/return_15m_forward/sy

,symbol,horizon_minutes,date,rows,key,status
0,ADAUSDT,15,2020-02-01,1439,features/return_15m_forward/symbol=ADAUSDT/int...,uploaded
1,ADAUSDT,15,2020-02-02,1440,features/return_15m_forward/symbol=ADAUSDT/int...,uploaded
2,ADAUSDT,15,2020-02-03,1440,features/return_15m_forward/symbol=ADAUSDT/int...,uploaded
3,ADAUSDT,15,2020-02-04,1440,features/return_15m_forward/symbol=ADAUSDT/int...,uploaded
4,ADAUSDT,15,2020-02-05,1440,features/return_15m_forward/symbol=ADAUSDT/int...,uploaded
...,...,...,...,...,...,...
6574,ADAUSDT,25,2026-01-28,1440,features/return_25m_forward/symbol=ADAUSDT/int...,uploaded
6575,ADAUSDT,25,2026-01-29,1440,features/return_25m_forward/symbol=ADAUSDT/int...,uploaded
6576,ADAUSDT,25,2026-01-30,1440,features/return_25m_forward/symbol=ADAUSDT/int...,uploaded
6577,ADAUSDT,25,2026-01-31,1440,features/return_25m_forward/symbol=ADAUSDT/int...,uploaded


In [12]:
df = pd.read_parquet(r'C:\projects\binance-dowloader-3.0\data (6).parquet')
df

,date,symbol,timestamp,return_1m
0,2020-02-01,ADAUSDT,2020-02-01 00:01:00+00:00,-0.000743
1,2020-02-01,ADAUSDT,2020-02-01 00:02:00+00:00,0.000186
2,2020-02-01,ADAUSDT,2020-02-01 00:03:00+00:00,-0.001859
3,2020-02-01,ADAUSDT,2020-02-01 00:04:00+00:00,-0.000745
4,2020-02-01,ADAUSDT,2020-02-01 00:05:00+00:00,0.000932
...,...,...,...,...
1434,2020-02-01,ADAUSDT,2020-02-01 23:55:00+00:00,-0.000356
1435,2020-02-01,ADAUSDT,2020-02-01 23:56:00+00:00,0.000535
1436,2020-02-01,ADAUSDT,2020-02-01 23:57:00+00:00,-0.000356
1437,2020-02-01,ADAUSDT,2020-02-01 23:58:00+00:00,0.001604
